In [1]:
import os
import numpy as np
import pandas as pd
%pip install pulp
import pulp
import time

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Loading processed data

current_dir = os.getcwd()
data_dir = os.path.join(os.path.dirname(current_dir), 'data', 'processed')

csv_file_path = os.path.join(data_dir, "cleaned_delivery_orders.csv")
npz_file_path = os.path.join(data_dir, "vrp_mathematical_matrices.npz")

df = pd.read_csv(csv_file_path)
print("DataFrame loaded successfully.")

matrices = np.load(npz_file_path, allow_pickle=True)
print("Matrices loaded successfully.And available matrices are:", matrices.files)

cargo_weights=matrices['cargo_weights']
order_values=matrices['order_values']
service_times=matrices['service_times']
window_starts=matrices['window_starts']
window_ends=matrices['window_ends']
distance_matrix_km=matrices['distance_matrix_km']
time_cost_matrix=matrices['time_cost_matrix']
vehicle_capacities = matrices['vehicle_capacities']
vehicle_shifts_min = matrices['vehicle_shifts_min']

print('All matrices loaded successfully.')

DataFrame loaded successfully.
Matrices loaded successfully.And available matrices are: ['cargo_weights', 'order_values', 'service_times', 'window_starts', 'window_ends', 'distance_matrix_km', 'time_cost_matrix', 'vehicle_capacities', 'vehicle_shifts_min']
All matrices loaded successfully.


In [3]:
# scalability slicer
N = 20
slice_limit = N + 1

# Slice 1D arrays
sliced_cargo_weights = cargo_weights[:slice_limit]
sliced_order_values = order_values[:slice_limit]
sliced_service_times = service_times[:slice_limit]
sliced_window_starts = window_starts[:slice_limit]
sliced_window_ends = window_ends[:slice_limit]

# Slice 2D arrays
sliced_distance_matrix_km = distance_matrix_km[:slice_limit, :slice_limit]
sliced_time_cost_matrix = time_cost_matrix[:slice_limit, :slice_limit]

# Slice the DataFrame
sliced_df = df.iloc[:slice_limit].copy()

print("Sliced DataFrame and matrices successfully.")
print("Sliced DataFrame shape:", sliced_df.shape)
print("Sliced cargo_weights shape:", sliced_cargo_weights.shape)
print("Sliced order_values shape:", sliced_order_values.shape)
print("Sliced service_times shape:", sliced_service_times.shape)
print("Sliced window_starts shape:", sliced_window_starts.shape)
print("Sliced window_ends shape:", sliced_window_ends.shape)
print("Sliced distance_matrix_km shape:", sliced_distance_matrix_km.shape)
print("Sliced time_cost_matrix shape:", sliced_time_cost_matrix.shape)


Sliced DataFrame and matrices successfully.
Sliced DataFrame shape: (21, 12)
Sliced cargo_weights shape: (21,)
Sliced order_values shape: (21,)
Sliced service_times shape: (21,)
Sliced window_starts shape: (21,)
Sliced window_ends shape: (21,)
Sliced distance_matrix_km shape: (21, 21)
Sliced time_cost_matrix shape: (21, 21)


In [4]:
# Initialize the Engine
model = pulp.LpProblem("VRP_Profit_Maximization", pulp.LpMaximize)
print("Model initialized successfully.")

# Define dimensions
num_vehicles = len(vehicle_capacities) 
K = range(num_vehicles)
V = range(slice_limit)  # Number of orders (including depot)
K = range(num_vehicles)  # Number of vehicles

x = pulp.LpVariable.dicts("route", (V, V, K), cat=pulp.LpBinary)
print("Decision variables defined successfully.")

# Create the routing variable
y = pulp.LpVariable.dicts("select", (V, K), cat=pulp.LpBinary)
print("Routing variables defined successfully.")


# Create time variable
t = pulp.LpVariable.dicts("time", (V, K), lowBound=0, cat=pulp.LpContinuous)
print("Time variables defined successfully.")

print("All variables defined successfully.")
print("Number of orders (including depot):", slice_limit)
print("Number of vehicles:", num_vehicles)


Model initialized successfully.
Decision variables defined successfully.
Routing variables defined successfully.
Time variables defined successfully.
All variables defined successfully.
Number of orders (including depot): 21
Number of vehicles: 20


In [5]:
# Force all arrays to be pure floats so PuLP can perform algebra on them
sliced_cargo_weights = sliced_cargo_weights.astype(float)
sliced_order_values = sliced_order_values.astype(float)
sliced_service_times = sliced_service_times.astype(float)
sliced_window_starts = sliced_window_starts.astype(float)
sliced_window_ends = sliced_window_ends.astype(float)
sliced_distance_matrix_km = sliced_distance_matrix_km.astype(float)
sliced_time_cost_matrix = sliced_time_cost_matrix.astype(float)
sliced_priorities = df['Priority_Level'].to_numpy()[:slice_limit].astype(int)
num_vehicles = vehicle_capacities = matrices['vehicle_capacities']
vehicle_shifts_min = matrices['vehicle_shifts_min']


num_vehicles = len(vehicle_capacities)
V = range(slice_limit) 
K = range(num_vehicles)


#  MODEL & VARIABLE INITIALIZATION
model = pulp.LpProblem("VRP_Profit_Maximization", pulp.LpMaximize)

x = pulp.LpVariable.dicts("route", (V, V, K), cat=pulp.LpBinary)
y = pulp.LpVariable.dicts("select", (V, K), cat=pulp.LpBinary)
t = pulp.LpVariable.dicts("time", (V, K), lowBound=0, cat=pulp.LpContinuous)


# Define the objective function
penalty_value = 10000 # Penalty for not serving an order

objective = (
    pulp.lpSum(y[i][k] * sliced_order_values[i] for i in V for k in K) -
    pulp.lpSum(x[i][j][k] * sliced_time_cost_matrix[i][j] for i in V for j in V for k in K)
    
    - pulp.lpSum(
        penalty_value * (1 - pulp.lpSum(y[i][k] for k in K)) for i in V if  sliced_priorities[i] == 5 and i != 0
    )
)
model += objective, "Maximize_Profit"
print("Objective function defined successfully.")


# Mathematical Constraints
M = 1000000

for k in K:
    model += pulp.lpSum(y[i][k] * sliced_cargo_weights[i] for i in V) <= vehicle_capacities[k], f"Capacity_Veh_{k}"
    
    model += (
        pulp.lpSum(y[i][k] * sliced_service_times[i] for i in V) +
        pulp.lpSum(x[i][j][k] * sliced_time_cost_matrix[i][j] for i in V for j in V if i != j)
    ) <= vehicle_shifts_min[k], f"Shift_Time_Veh_{k}"
    
    
    for i in V:
        model += pulp.lpSum(x[i][j][k] for j in V if j != i) == y[i][k], f"Leave_order_{i}_vehicle_{k}"
        model += pulp.lpSum(x[j][i][k] for j in V if j != i) == y[i][k], f"Enter_order_{i}_vehicle_{k}"
        
    
# Add time window constraints
for k in K:
    for i in V:
        model += t[i][k] >= sliced_window_starts[i] * y[i][k], f"Time_Window_Start_Order_{i}_Vehicle_{k}"
        model += t[i][k] <= sliced_window_ends[i] * y[i][k] + M * (1 - y[i][k]), f"Time_Window_End_Order_{i}_Vehicle_{k}"
        
        # link routing and time variables
        for j in V:
            if i != 0 and j != 0 and i != j:
                model += t[j][k] >= t[i][k] + sliced_service_times[i] + sliced_time_cost_matrix[i][j] - M * (1 - x[i][j][k]), f"Time_Link_Order_{i}_to_{j}_Vehicle_{k}"
                
#for at least one vehicle leave the depot
model += pulp.lpSum(y[0][k] for k in K) >= 1, "At_least_one_vehicle_leaves_depot"
           
print("All constraints added successfully.")

Objective function defined successfully.
All constraints added successfully.


In [6]:
# Start the Benchmarking Timer
start_time = time.time()

print("Starting to solve the model...")

time_limit_sec = 300
model.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=time_limit_sec))

# Stop the timer and calculate the elapsed time
end_time = time.time()
elapsed_time = end_time - start_time

# log and calculate the KPIs
status = pulp.LpStatus[model.status]
print('Execution Results')
print('Status:', status)
print('Execution Time (seconds):', elapsed_time)

calculated_revenue = 0.0
route_nodes = []
deferred_nodes = []

if status in ('Optimal', 'Not Solved'):
    calculated_revenue = sum(
        sliced_order_values[i] for i in V for k in K if pulp.value(y[i][k]) == 1
    )
    print('Calculated Revenue LKR:', calculated_revenue)

    for i in V:
        if i == 0:
            continue
        is_visited = sum((pulp.value(y[i][k]) or 0) for k in K)

        if is_visited > 0.5:
            route_nodes.append(i)
        else:
            deferred_nodes.append(i)
    print(f'Sucessfully Routed Orders: {route_nodes}')
    print(f'Deferred Orders: {deferred_nodes}')
else:
    print('No optimal solution found. Please check the model and constraints.')
    print('See the diagnostic cell below to investigate which constraint is causing infeasibility.')

mip_results = {
    'Revenue': calculated_revenue,
    'Execution_Time': elapsed_time,
    'Scalability': slice_limit - 1,
    'Status': status
}

import json
with open('mip_results.json', 'w') as f:
    json.dump(mip_results, f)
print("MIP results saved to mip_results.json")

Starting to solve the model...
Execution Results
Status: Optimal
Execution Time (seconds): 300.27858328819275
Calculated Revenue LKR: 1727700.0
Sucessfully Routed Orders: [2, 3, 5, 8, 10, 11, 17, 18]
Deferred Orders: [1, 4, 6, 7, 9, 12, 13, 14, 15, 16, 19, 20]
MIP results saved to mip_results.json


In [7]:
# Write the model to a file to inspect the equations
model.writeLP("vrp_debug.lp")
print("Model written to vrp_debug.lp. Open this file to see the constraints.")

# Print the number of constraints to ensure they were actually added
print(f"Total constraints in model: {len(model.constraints)}")

Model written to vrp_debug.lp. Open this file to see the constraints.
Total constraints in model: 9321


In [8]:
def solve_mip(N, df, matrices, time_limit=120):
    """
    Scalability analysis - REAL runs across increasing problem size N.
    """
    slice_limit = N + 1
    cargo_weights = matrices['cargo_weights'][:slice_limit].astype(float)
    order_values = matrices['order_values'][:slice_limit].astype(float)
    service_times = matrices['service_times'][:slice_limit].astype(float)
    window_starts = matrices['window_starts'][:slice_limit].astype(float)
    window_ends = matrices['window_ends'][:slice_limit].astype(float)
    time_cost_matrix = matrices['time_cost_matrix'][:slice_limit, :slice_limit].astype(float)
    priorities = df['Priority_Level'].to_numpy()[:slice_limit].astype(int)
    vehicle_capacities = matrices['vehicle_capacities']
    vehicle_shifts_min = matrices['vehicle_shifts_min']
    
    num_vehicles = len(vehicle_capacities)
    V, K = range(slice_limit), range(num_vehicles)
    
    model = pulp.LpProblem("VRP_Profit_Maximization", pulp.LpMaximize)
    x = pulp.LpVariable.dicts("route", (V, V, K), cat=pulp.LpBinary)
    y = pulp.LpVariable.dicts("select", (V, K), cat=pulp.LpBinary)
    t = pulp.LpVariable.dicts("time", (V, K), lowBound=0, cat=pulp.LpContinuous)
    
    penalty_value = 10000
    model += (  pulp.lpSum(y[i][k] * order_values[i] for i in V for k in K) -
        pulp.lpSum(x[i][j][k] * time_cost_matrix[i][j] for i in V for j in V for k in K)
        - pulp.lpSum(penalty_value * (1 - pulp.lpSum(y[i][k] for k in K)) for i in V if priorities[i] == 5 and i != 0)
    ), "Maximize_Profit"

    M = 1000000
    for k in K:
        model += pulp.lpSum(y[i][k] * cargo_weights[i] for i in V) <= vehicle_capacities[k], f"Capacity_Veh_{k}"
        model += (pulp.lpSum(y[i][k] * service_times[i] for i in V) +
                   pulp.lpSum(x[i][j][k] * time_cost_matrix[i][j] for i in V for j in V if i != j)) <= vehicle_shifts_min[k], f"Shift_Time_Veh_{k}"
        for i in V:
            model += pulp.lpSum(x[i][j][k] for j in V if j != i) == y[i][k], f"Leave_{i}_{k}"
            model += pulp.lpSum(x[j][i][k] for j in V if j != i) == y[i][k], f"Enter_{i}_{k}"

    for k in K:
        for i in V:
            model += t[i][k] >= window_starts[i] * y[i][k], f"TW_Start_{i}_{k}"
            model += t[i][k] <= window_ends[i] * y[i][k] + M * (1 - y[i][k]), f"TW_End_{i}_{k}"
            for j in V:
                if i != 0 and j != 0 and i != j:
                    model += t[j][k] >= t[i][k] + service_times[i] + time_cost_matrix[i][j] - M * (1 - x[i][j][k]), f"Link_{i}_{j}_{k}"

    model += pulp.lpSum(y[0][k] for k in K) >= 1, "AtLeastOneVehicle"

    start = time.time()
    model.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=time_limit))
    elapsed = time.time() - start
    status = pulp.LpStatus[model.status]
    
    revenue, routed, deferred = 0.0, [], []
    if status in ('Optimal', 'Not Solved'):
        revenue = sum(order_values[i] for i in V for k in K if pulp.value(y[i][k]) == 1)
        for i in V:
            if i == 0:
                continue
            visited = sum((pulp.value(y[i][k]) or 0) for k in K)
            (routed if visited > 0.5 else deferred).append(i)
            

    return {'N': N, 'Status': status, 'Revenue': float(revenue), 'Execution_Time': elapsed,
            'Routed': len(routed), 'Deferred': len(deferred), 'Time_Limited': elapsed >= time_limit - 0.5}


results_log = []
for N in [5, 10, 15, 20]:
    print(f"Solving MIP for N={N} orders...")
    r = solve_mip(N, df, matrices, time_limit=120)
    print(r)
    results_log.append(r)
    
scalability_df = pd.DataFrame(results_log)
display(scalability_df)

with open('mip_scalability_results.json', 'w') as f:
    json.dump(results_log, f)
print("MIP scalability results saved to mip_scalability_results.json")

Solving MIP for N=5 orders...
{'N': 5, 'Status': 'Optimal', 'Revenue': 2175400.0, 'Execution_Time': 12.018566846847534, 'Routed': 4, 'Deferred': 1, 'Time_Limited': False}
Solving MIP for N=10 orders...
{'N': 10, 'Status': 'Optimal', 'Revenue': 2315400.0, 'Execution_Time': 119.91754651069641, 'Routed': 8, 'Deferred': 2, 'Time_Limited': True}
Solving MIP for N=15 orders...
{'N': 15, 'Status': 'Optimal', 'Revenue': 2535300.0, 'Execution_Time': 119.39709401130676, 'Routed': 10, 'Deferred': 5, 'Time_Limited': False}
Solving MIP for N=20 orders...
{'N': 20, 'Status': 'Optimal', 'Revenue': 1727700.0, 'Execution_Time': 118.40927362442017, 'Routed': 8, 'Deferred': 12, 'Time_Limited': False}


,N,Status,Revenue,Execution_Time,Routed,Deferred,Time_Limited
0,5,Optimal,2175400.0,12.018567,4,1,False
1,10,Optimal,2315400.0,119.917547,8,2,True
2,15,Optimal,2535300.0,119.397094,10,5,False
3,20,Optimal,1727700.0,118.409274,8,12,False


MIP scalability results saved to mip_scalability_results.json


In [9]:
# Compare your actual matrix values against what I tested
print("time_cost_matrix[0][:6] (depot -> first 5 orders, minutes):")
print(matrices['time_cost_matrix'][0][:6])

print("\nwindow_starts[:6]:", matrices['window_starts'][:6])
print("window_ends[:6]:", matrices['window_ends'][:6])
print("vehicle_shifts_min (unique):", sorted(set(matrices['vehicle_shifts_min'])))

print("\nSanity: is time_cost_matrix identical to distance_matrix_km? (should be False now)")
print(np.allclose(matrices['time_cost_matrix'], matrices['distance_matrix_km']))

time_cost_matrix[0][:6] (depot -> first 5 orders, minutes):
[  0.         551.43771777 241.65406233 148.50991869  21.22131342
  82.83668309]

window_starts[:6]: [  0 780 480 780 780   0]
window_ends[:6]: [   0 1020  720 1020 1020 1439]
vehicle_shifts_min (unique): [np.int64(540), np.int64(660)]

Sanity: is time_cost_matrix identical to distance_matrix_km? (should be False now)
False
